In [ ]:
import json, os, glob, tqdm
from PIL import Image
import numpy as np
import subprocess
import imageio
import numpy as np

def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

def face_segment(segment_part, mask_path):
    face_segment_anno = imageio.v2.imread(mask_path)

    face_segment_anno = np.array(face_segment_anno)
    bg = (face_segment_anno == 0)
    skin = (face_segment_anno == 1)
    l_brow = (face_segment_anno == 2)
    r_brow = (face_segment_anno == 3)
    l_eye = (face_segment_anno == 4)
    r_eye = (face_segment_anno == 5)
    eye_g = (face_segment_anno == 6)
    l_ear = (face_segment_anno == 7)
    r_ear = (face_segment_anno == 8)
    ear_r = (face_segment_anno == 9)
    nose = (face_segment_anno == 10)
    mouth = (face_segment_anno == 11)
    u_lip = (face_segment_anno == 12)
    l_lip = (face_segment_anno == 13)
    neck = (face_segment_anno == 14)
    neck_l = (face_segment_anno == 15)
    cloth = (face_segment_anno == 16)
    hair = (face_segment_anno == 17)
    hat = (face_segment_anno == 18)
    face = np.logical_or.reduce((skin, l_brow, r_brow, l_eye, r_eye, eye_g, l_ear, r_ear, ear_r, nose, mouth, u_lip, l_lip))

    if segment_part == 'faceseg_bg':
        seg_m = bg
    elif segment_part == 'faceseg_fg':
        seg_m = ~bg
    else: raise NotImplementedError(f"Segment part: {segment_part} is not found!")
    
    out = seg_m
    return out
            
def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

def blending_mask(img_path, mask_path, hdr_from_bg_path):
    if isinstance(mask_path, np.ndarray):
        mask = mask_path / 255.0
    else:
        mask = imageio.v2.imread(mask_path) / 255.0 #[256, 256]
        
    if isinstance(img_path, np.ndarray):
        img = img_path / 255.0
    else:
        img = imageio.v2.imread(img_path).astype(np.float32) / 255.0
        
    blurred_mask = cv2.GaussianBlur(mask, (3, 3), 0)
    # Apply erosion
    kernel = np.ones((3,3),np.uint8)
    eroded_mask = cv2.erode(blurred_mask, kernel, iterations = 1)
    mask = eroded_mask [..., np.newaxis]
    
    bg = imageio.v2.imread(hdr_from_bg_path).astype(np.float32) / 255.0
    # alpha blending
    out = img * mask + bg * (1 - mask)
    out = (out * 255).astype(np.uint8)
    return out

In [60]:
import numpy as np
import torch as th
import json, os, glob
import imageio
import cv2
import multiprocessing as mp
import matplotlib.pyplot as plt
from PIL import Image

# Azimuth = Axis 1
method = ["neural_gaffer_azimuth", "difareli++_axis1_color_SD75_0.8C_Lmax10"]
# method = ["neural_gaffer_azimuth", "difareli++_axis1_color_SD50_0.8C_Lmax10"]
meta = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/MajorRevision/hdr/hdr_finale_axis=1.json"))
sample = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/all_rotateSH.json"))
# sample = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/aj_ake_samples.json"))
out_dir = "./vids_hdr/"
os.makedirs(out_dir, exist_ok=True)
# default_fidx = [33, 26, 17, 6]  # 0-59
# hdr_map = ["012_hdrmaps_com_free_2K", "064_hdrmaps_com_free_2K", "117_hdrmaps_com_free_2K", "125_hdrmaps_com_free_2K", "128_hdrmaps_com_free_2K"]
hdr_map = ["125_hdrmaps_com_free_2K", "064_hdrmaps_com_free_2K"]
# hdr_map = ["064_hdrmaps_com_free_2K"]
# hdr_map = ["125_hdrmaps_com_free_2K"]
default_fidx = [[26, 17, 6], [20, 11, 58]]  # 0-59
# focus_src = [f'{f}.jpg' for f in [64576, 64137, 69157, 63825, 64240, 62514, 60006, 65267, 65857, 60609]]
focus_src = [f'{f}.jpg' for f in [64240]]
for pid, dat in sample['pair'].items():
    src = dat['src']
    dst = dat['dst']
    if src not in focus_src:
        continue
    
    for hidx, hdr in enumerate(hdr_map):
        hdr_fidx = dat.get(f'hdr={hdr}', default_fidx[hidx])
        hdr_fidx = [i for i in range(hdr_fidx[0], 59, 1)] + [i for i in range(0, hdr_fidx[0], 1)]
        print(hdr_fidx)
        out_combined = []
        fail = False
        for m in method:
            if m == "neural_gaffer_azimuth":
                path = meta[m]['res_dir']
                frame_path = f"{path}/{src.split('.')[0]}/{hdr}/"
                frames = sorted(glob.glob(f"{frame_path}/pred_*.png"))
                if len(frames) < 1:
                    print(f"[#] Skip {pid}-{src.split('.')[0]} for {m} since no result is found on {hdr}.")
                    fail = True
                    break
                # frames = [f'{frame_path}/pred_{58-i:04d}.png' for i in hdr_fidx]
                frames = [f'{frame_path}/pred_{58-i:04d}.png' for i in hdr_fidx]
                # target_hdr = [f"{path}/{src.split('.')[0]}/{hdr}/target_{59-i-1:04d}.png" for i in hdr_fidx]
                target_hdr = [f"/data/mint/DPM_Dataset/Dataset_For_Baseline/NeuralGaffer/lighting_for_visualization/rotate_sh_axis=1_azimuth/{hdr}/background/{58-i}.png" for i in hdr_fidx]
                target_ldr_orig = [f"/data/mint/DPM_Dataset/Dataset_For_Baseline/NeuralGaffer/lighting_for_inference/rotate_sh_axis=1_azimuth/{hdr}/LDR_original_size/{58-i}.png" for i in hdr_fidx]
                mask = f"/data/mint/DPM_Dataset/Dataset_For_Baseline/NeuralGaffer/input_subject_finale/preprocessed/mask/{src.split('.')[0]}.png"
            elif m == "difareli++_axis1_color_SD75_0.8C_Lmax10":
            # elif m == "difareli++_axis1_color_SD50_0.8C_Lmax10":
                path = meta[m]['res_dir']
                frame_path = f"{path}/{hdr}/src={src}/dst={dst}/Lerp_1000/n_frames=60/"
                frames = sort_by_frame(glob.glob(f"{frame_path}/res_frame*.png"))[1:]
                if len(frames) < 1:
                    print(f"Skip {pid}-{src.split('.')[0]} for {m} since no result is found on {hdr}.")
                    fail = True
                    break
                frames = [f'{frame_path}/res_frame{i+1}.png' for i in hdr_fidx]
                render_frames = [f'{frame_path}/dst_ren_frame{i+1}.png' for i in hdr_fidx]    # 1-59 (0 is src, n=58)
                shadow_frames = [f'{frame_path}/dst_shadm_shad_frame{i+1}.png' for i in hdr_fidx]
                assert len(shadow_frames) == len(render_frames)
                cond = [np.concatenate([imageio.v2.imread(render_frames[i]), imageio.v2.imread(shadow_frames[i])], axis=1) for i in range(len(render_frames))]
                cond = np.stack(cond)
                
                # target_hdr = [f"{meta['neural_gaffer_azimuth']['res_dir']}/{src.split('.')[0]}/{hdr}/target_{59-i-1:04d}.png" for i in hdr_fidx]
                target_hdr = [f"/data/mint/DPM_Dataset/Dataset_For_Baseline/NeuralGaffer/lighting_for_visualization/rotate_sh_axis=1_azimuth/{hdr}/background/{58-i}.png" for i in hdr_fidx]
                target_ldr_orig = [f"/data/mint/DPM_Dataset/Dataset_For_Baseline/NeuralGaffer/lighting_for_inference/rotate_sh_axis=1_azimuth/{hdr}/LDR_original_size/{58-i}.png" for i in hdr_fidx]
                mask = f"/data/mint/DPM_Dataset/ffhq_256_with_anno/face_segment_with_pupil/valid/anno/anno_{src.split('.')[0]}.png"
                mask = face_segment('faceseg_fg', mask) * 1.0
                
            # Reuse the target hdr as background for difareli++ too, but need to rescale first
            if "difareli++" in m:
                frames_tmp = [imageio.v2.imread(f) for f in frames] # 0 - 255
                mask_tmp = mask
                out_frames = []
                out_mask = []
                for i in range(len(frames_tmp)):
                    proc_img = frames_tmp[i]
                    proc_img = np.concatenate([proc_img, mask_tmp[..., np.newaxis] * 255], axis=-1)
                    
                    x, y, w, h = cv2.boundingRect((mask_tmp*255).astype(np.uint8))
                    max_size = max(w, h)
                    ratio = 0.75
                    side_len = int(max_size / ratio)
                    padded_image = np.zeros((side_len, side_len, 4), dtype=np.uint8)
                    center = side_len//2
                    padded_image[center-h//2:center-h//2+h, center-w//2:center-w//2+w] = proc_img[y:y+h, x:x+w]
                    rgba = np.array(Image.fromarray(padded_image).resize((256, 256), Image.LANCZOS))
                    # rgba is 0 - 255 for each channel
                    rgba_arr = np.array(rgba) / 255.0   # normalized to 0 - 1
                    rgb = rgba_arr[...,:3] * rgba_arr[...,-1:] + (1 - rgba_arr[...,-1:])    # 256, 256, 3; 0-1
                    mask = rgba_arr[...,-1:]    # 256, 256, 1; 0-1
                    out_frames.append((rgb * 255).astype(np.uint8))
                    out_mask.append((mask[...,0] * 255).astype(np.uint8))
                frames = out_frames
                mask = out_mask[0]
            
            with mp.Pool(5) as p:
                out_frames = p.starmap(blending_mask, (zip(frames, [mask]*len(frames), target_hdr)))
                
            out_frames = np.stack(out_frames)
            out_combined.append(out_frames)

            # Reduce target_ldr_orig to 256 height and preserve aspect ratio
            imgs = []
            for f in target_ldr_orig:
                img = imageio.imread(f)
                h, w = img.shape[:2]
                scale = 256.0 / h
                resized = cv2.resize(img, (int(w * scale), 256), interpolation=cv2.INTER_LANCZOS4)
                imgs.append(resized)
            
            target_ldr_orig = np.stack(imgs)
            
        if not fail:
            # Save images
            for im, m in enumerate(method):
                out_frames = out_combined[im]
                out_frame_path = f'{out_dir}/{hdr}/{src.split(".")[0]}/{m}/'
                os.makedirs(out_frame_path, exist_ok=True)
                for i in range(out_frames.shape[0]):
                    imageio.imsave(f"{out_frame_path}/res_frame_{i:04d}.png", out_frames[i])
                # Create videos
                cmd = f'ffmpeg -y -r 24 -i ./vids_hdr/{hdr}/{src.split(".")[0]}/{m}/res_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids_hdr/{hdr}/{src.split(".")[0]}/res_{m}.mp4'
                os.system(cmd)
            
            
            for i in range(cond.shape[0]):
                # Resize to have width = 256
                Image.fromarray(cond[i]).resize((256, int(cond.shape[1] * 256 / cond.shape[2])), Image.LANCZOS).save(f"{out_dir}/{hdr}/{src.split('.')[0]}/{method[1]}/cond_{i:04d}.png")

            cmd_cond = f'ffmpeg -y -r 24 -i ./vids_hdr/{hdr}/{src.split(".")[0]}/{method[1]}/cond_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids_hdr/{hdr}/{src.split(".")[0]}/cond.mp4'
            os.system(cmd_cond)
            
            # Create combined video stack vertically
            cmd_stack = f'ffmpeg -y -i ./vids_hdr/{hdr}/{src.split(".")[0]}/res_{method[0]}.mp4 -i ./vids_hdr/{hdr}/{src.split(".")[0]}/res_{method[1]}.mp4 -i ./vids_hdr/{hdr}/{src.split(".")[0]}/cond.mp4 -filter_complex vstack=inputs=3 ./vids_hdr/{hdr}/{src.split(".")[0]}/res_combined.mp4'
            os.system(cmd_stack)
                
            # Save ldr frame
            ldr_frame_path = f'{out_dir}/{hdr}/LDR_original_size/'
            os.makedirs(ldr_frame_path, exist_ok=True)
            for i in range(target_ldr_orig.shape[0]):
                imageio.imsave(f"{ldr_frame_path}/LDR_original_size_{i:04d}.png", target_ldr_orig[i])
            # Create videos   
            cmd_hdr = f'ffmpeg -y -r 24 -i ./vids_hdr/{hdr}/LDR_original_size/LDR_original_size_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids_hdr/{hdr}/LDR_original_size.mp4'
            os.system(cmd_hdr)
            

[26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


/tmp/ipykernel_2593788/2356387769.py:111: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(f)
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enabl

[20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


frame=   59 fps=0.0 q=-1.0 Lsize=     169kB time=00:00:02.33 bitrate= 594.3kbits/s speed=5.63x    
video:168kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.865264%
[libx264 @ 0x55d622040a80] frame I:1     Avg QP:15.07  size: 40966
[libx264 @ 0x55d622040a80] frame P:23    Avg QP:15.94  size:  4543
[libx264 @ 0x55d622040a80] frame B:35    Avg QP:22.85  size:   735
[libx264 @ 0x55d622040a80] consecutive B-frames:  1.7% 40.7% 50.8%  6.8%
[libx264 @ 0x55d622040a80] mb I  I16..4:  0.8% 29.1% 70.1%
[libx264 @ 0x55d622040a80] mb P  I16..4:  0.1%  1.0%  2.3%  P16..4: 50.2%  6.6%  9.4%  0.0%  0.0%    skip:30.4%
[libx264 @ 0x55d622040a80] mb B  I16..4:  0.0%  0.0%  0.0%  B16..8: 33.1%  1.7%  0.6%  direct: 2.9%  skip:61.7%  L0:62.2% L1:35.4% BI: 2.3%
[libx264 @ 0x55d622040a80] 8x8 transform intra:29.7% inter:29.2%
[libx264 @ 0x55d622040a80] direct mvs  spatial:88.6% temporal:11.4%
[libx264 @ 0x55d622040a80] coded y,uvDC,uvAC intra: 98.2% 79.4% 59.3% inter: 11.9% 1

# Generate the full grid

In [62]:
import numpy as np
import torch as th
import json, os, glob
import imageio
import cv2
import multiprocessing as mp
import matplotlib.pyplot as plt
from PIL import Image

# Azimuth = Axis 1
method = ["neural_gaffer_azimuth", "difareli++_axis1_color_SD75_0.8C_Lmax10"]
# method = ["neural_gaffer_azimuth", "difareli++_axis1_color_SD50_0.8C_Lmax10"]
meta = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/MajorRevision/hdr/hdr_finale_axis=1.json"))
# sample = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/all_rotateSH.json"))
sample = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/aj_ake_samples.json"))
out_dir = "./vids_hdr/"
os.makedirs(out_dir, exist_ok=True)
# default_fidx = [33, 26, 17, 6]  # 0-59
# hdr_map = ["012_hdrmaps_com_free_2K", "064_hdrmaps_com_free_2K", "117_hdrmaps_com_free_2K", "125_hdrmaps_com_free_2K", "128_hdrmaps_com_free_2K"]
hdr_map = ["125_hdrmaps_com_free_2K", "064_hdrmaps_com_free_2K"]
# hdr_map = ["064_hdrmaps_com_free_2K"]
# hdr_map = ["125_hdrmaps_com_free_2K"]
default_fidx = [[26, 17, 6], [20, 11, 58]]  # 0-59
# focus_src = [f'{f}.jpg' for f in [64576, 64137, 69157, 63825, 64240, 62514, 60006, 65267, 65857, 60609]]
focus_src = [f'{f}.jpg' for f in [64576, 64137, 64240, 62514]]

for hidx, hdr in enumerate(hdr_map):
    out_grid = []
    for src in focus_src:
        vid_path = f"./vids_hdr/{hdr}/{src.split('.')[0]}/res_combined.mp4"
        out_grid.append(vid_path)
        
    # Generate the full grid by stack horizontally (With white space in between, 10 pixels)
    # spacing in pixels between videos
    gap = 10  # white gap in pixels
    N = len(out_grid) # Assuming out_grid is the list of video paths

    # 1. Generate Padding Filters
    padding_filters = []
    for i in range(N - 1):
        padding_filters.append(f"[{i}:v]pad=iw+10:ih:0:0:color=white[p{i}];")

    padding_string = "".join(padding_filters)

    # 2. Generate hstack Inputs
    # (p0)(p1)...(p(N-2)) + (N-1:v)
    hstack_inputs = "".join([f"[p{i}]" for i in range(N - 1)])
    hstack_inputs += f"[{N-1}:v]"

    # 3. Combine Filter Complex
    filter_complex = f"{padding_string}{hstack_inputs}hstack=inputs={N}"

    # 4. Final Command
    cmd_grid = (
        f"ffmpeg -y "
        + " ".join([f"-i {p}" for p in out_grid])
        + f" -filter_complex \"{filter_complex}\" "
        + f"-pix_fmt yuv420p -crf 13 ./vids_hdr/{hdr}/res_combined_grid.mp4"
    )
    os.system(cmd_grid)


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab